# Tutorial: Sound minimum accuracy limits

This notebook shows the difference between using one shared `min_acc_limit` and setting separate limits for soft and hard accuracy.

The short version:

- `min_acc_limit` is the backward-compatible fallback. If you do not set explicit limits, it is used for both soft and hard accuracy.
- `min_soft_acc_limit` controls the differentiable soft-accuracy constraint used during Rashomon-set optimization.
- `min_hard_acc_limit` controls the hard certificate gate. This is the one to keep strict when you need a sound safety or validity guarantee.

A common useful pattern is to relax the soft surrogate while keeping the hard requirement strict, for example `min_soft_acc_limit=0.5` and `min_hard_acc_limit=1.0`.

## Setup

These cells are intentionally lightweight. They use toy interval logits to illustrate the semantics without loading a trained LunarLander policy or running a full Rashomon optimization.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import torch

try:
    import pandas as pd
except ImportError:  # The notebook still works without pandas.
    pd = None


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "core" / "src").exists() and (candidate / "experiments").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
CORE_ROOT = REPO_ROOT / "core"
if str(CORE_ROOT) not in sys.path:
    sys.path.insert(0, str(CORE_ROOT))

from src.IntervalTensor import IntervalTensor
import src.verification.verify as verify


def table(rows):
    if pd is None:
        return rows
    return pd.DataFrame(rows)


torch.set_printoptions(precision=4, sci_mode=False)
REPO_ROOT

PosixPath('/vol/bitbucket/ma5923/_projects/CertifiedContinualLearning')

## Limit resolution

At the lower-level `src.interval_utils.compute_rashomon_set` API, explicit limits override the shared fallback:

```python
soft_limit = min_soft_acc_limit if min_soft_acc_limit is not None else min_acc_limit
hard_limit = min_hard_acc_limit if min_hard_acc_limit is not None else min_acc_limit
```

So lowering `min_acc_limit` to help the soft optimization also lowers the hard gate unless you explicitly set `min_hard_acc_limit`.

In [2]:
def resolve_lower_level_limits(
    *,
    min_acc_limit=0.85,
    min_soft_acc_limit=None,
    min_hard_acc_limit=None,
):
    soft = min_soft_acc_limit if min_soft_acc_limit is not None else min_acc_limit
    hard = min_hard_acc_limit if min_hard_acc_limit is not None else min_acc_limit
    if soft is None or hard is None:
        raise ValueError("At least one min-accuracy limit must be specified for each side.")
    return soft, hard


configs = [
    {
        "case": "shared only",
        "min_acc_limit": 0.50,
        "min_soft_acc_limit": None,
        "min_hard_acc_limit": None,
    },
    {
        "case": "separate soft and hard",
        "min_acc_limit": 0.50,
        "min_soft_acc_limit": 0.50,
        "min_hard_acc_limit": 1.00,
    },
    {
        "case": "soft fallback, explicit hard",
        "min_acc_limit": 0.50,
        "min_soft_acc_limit": None,
        "min_hard_acc_limit": 1.00,
    },
    {
        "case": "explicit soft, hard fallback",
        "min_acc_limit": 1.00,
        "min_soft_acc_limit": 0.50,
        "min_hard_acc_limit": None,
    },
]

rows = []
for cfg in configs:
    soft, hard = resolve_lower_level_limits(
        min_acc_limit=cfg["min_acc_limit"],
        min_soft_acc_limit=cfg["min_soft_acc_limit"],
        min_hard_acc_limit=cfg["min_hard_acc_limit"],
    )
    rows.append({**cfg, "resolved_soft_limit": soft, "resolved_hard_limit": hard})

table(rows)

,case,min_acc_limit,min_soft_acc_limit,min_hard_acc_limit,resolved_soft_limit,resolved_hard_limit
0,shared only,0.5,NaN,NaN,0.5,0.5
1,separate soft and hard,0.5,0.5,1.0,0.5,1.0
2,"soft fallback, explicit hard",0.5,NaN,1.0,0.5,1.0
3,"explicit soft, hard fallback",1.0,0.5,NaN,0.5,1.0


## One extra `IntervalTrainer` detail

`IntervalTrainer` has an additional `min_acc_increment` rule. With the default `min_acc_increment=0.05`, the trainer may resolve a requested limit relative to the current task accuracy. If you want literal values such as hard limit `1.0`, set `min_acc_increment=0.0` or `None` when constructing the trainer.

In [3]:
def resolve_interval_trainer_limit(*, base_limit, task_acc, min_acc_increment):
    if isinstance(base_limit, list):
        return base_limit
    if min_acc_increment and base_limit is not None:
        return min(max(task_acc - min_acc_increment, task_acc / 2), base_limit)
    if min_acc_increment:
        return max(task_acc - min_acc_increment, task_acc / 2)
    if base_limit is not None:
        return base_limit
    return None


def resolve_interval_trainer_limits(
    *,
    task_acc,
    min_acc_limit,
    min_soft_acc_limit=None,
    min_hard_acc_limit=None,
    min_acc_increment=0.05,
):
    base_soft = min_soft_acc_limit if min_soft_acc_limit is not None else min_acc_limit
    base_hard = min_hard_acc_limit if min_hard_acc_limit is not None else min_acc_limit
    return (
        resolve_interval_trainer_limit(
            base_limit=base_soft,
            task_acc=task_acc,
            min_acc_increment=min_acc_increment,
        ),
        resolve_interval_trainer_limit(
            base_limit=base_hard,
            task_acc=task_acc,
            min_acc_increment=min_acc_increment,
        ),
    )


task_acc = 1.0
rows = []
for increment in [0.05, 0.0, None]:
    soft, hard = resolve_interval_trainer_limits(
        task_acc=task_acc,
        min_acc_limit=0.50,
        min_soft_acc_limit=0.50,
        min_hard_acc_limit=1.00,
        min_acc_increment=increment,
    )
    rows.append(
        {
            "task_acc": task_acc,
            "min_acc_increment": increment,
            "requested_soft": 0.50,
            "requested_hard": 1.00,
            "trainer_resolved_soft": soft,
            "trainer_resolved_hard": hard,
        }
    )

table(rows)

,task_acc,min_acc_increment,requested_soft,requested_hard,trainer_resolved_soft,trainer_resolved_hard
0,1.0,0.05,0.5,1.0,0.5,0.95
1,1.0,0.00,0.5,1.0,0.5,1.00
2,1.0,NaN,0.5,1.0,0.5,1.00


## Toy interval logits

The next cells build four interval-logit examples. The target class is class `0` for every row.

For hard lower-bound accuracy, the verifier asks whether the target class still wins after putting the target logit at its lower bound and the non-target logit at its upper bound. For soft lower-bound accuracy, it measures the softmax probability mass on the target under the same worst-case logits.

That means soft accuracy can be useful as a smooth optimization signal while hard accuracy remains the certificate gate.

In [4]:
# Four samples, two classes. The true class is class 0 for every sample.
logits_l = torch.tensor(
    [
        [0.0, 0.0],  # not hard-certified: target lower is below class-1 upper
        [0.6, 0.0],  # hard-certified
        [0.2, 0.0],  # hard-certified
        [0.1, 0.0],  # hard-certified
    ]
)
logits_u = torch.tensor(
    [
        [0.0, 0.1],
        [0.6, 0.0],
        [0.2, 0.0],
        [0.1, 0.0],
    ]
)
targets = torch.zeros(4, dtype=torch.long)
logits = IntervalTensor(logits_l, logits_u)


def worst_case_logits_for_targets(logits_l, logits_u, targets):
    targets_one_hot = torch.nn.functional.one_hot(
        targets,
        num_classes=logits_l.shape[1],
    ).float()
    return targets_one_hot * logits_l + (1 - targets_one_hot) * logits_u


worst_case_logits = worst_case_logits_for_targets(logits_l, logits_u, targets)
hard_predictions = worst_case_logits.argmax(dim=1)
soft_probs_t10 = torch.softmax(worst_case_logits * 10, dim=1)[:, 0]

rows = []
for i in range(len(targets)):
    rows.append(
        {
            "sample": i,
            "target_lower": float(logits_l[i, 0]),
            "other_upper": float(logits_u[i, 1]),
            "hard_certified": bool(hard_predictions[i] == targets[i]),
            "soft_mass_T10": float(soft_probs_t10[i]),
        }
    )

table(rows)

,sample,target_lower,other_upper,hard_certified,soft_mass_T10
0,0,0.0,0.1,False,0.268941
1,1,0.6,0.0,True,0.997527
2,2,0.2,0.0,True,0.880797
3,3,0.1,0.0,True,0.731059


In [5]:
hard_acc = verify.bound_accuracy(logits, targets, lower=True)
soft_acc_t1 = verify.bound_soft_accuracy(logits, targets, T=1, lower=True)
soft_acc_t10 = verify.bound_soft_accuracy(logits, targets, T=10, lower=True)

table(
    [
        {"metric": "hard lower-bound accuracy", "value": float(hard_acc)},
        {"metric": "soft lower-bound accuracy, T=1", "value": float(soft_acc_t1)},
        {"metric": "soft lower-bound accuracy, T=10", "value": float(soft_acc_t10)},
    ]
)

,metric,value
0,hard lower-bound accuracy,0.750000
1,"soft lower-bound accuracy, T=1",0.548873
2,"soft lower-bound accuracy, T=10",0.719581


## Shared versus separate limits

With this toy set, the hard lower-bound accuracy is `0.75`: three out of four samples are certified. A shared `min_acc_limit=0.50` lets both the soft and hard checks pass. A separate configuration with `min_soft_acc_limit=0.50` and `min_hard_acc_limit=1.00` keeps the soft objective relaxed but rejects this interval because the hard certificate is not perfect.

In [6]:
resolved_configs = [
    {
        "configuration": "shared min_acc_limit=0.50",
        "soft_limit": 0.50,
        "hard_limit": 0.50,
    },
    {
        "configuration": "separate soft=0.50, hard=1.00",
        "soft_limit": 0.50,
        "hard_limit": 1.00,
    },
]

rows = []
for cfg in resolved_configs:
    soft_pass = bool(soft_acc_t10 >= cfg["soft_limit"])
    hard_pass = bool(hard_acc >= cfg["hard_limit"])
    rows.append(
        {
            **cfg,
            "soft_acc_T10": float(soft_acc_t10),
            "hard_acc": float(hard_acc),
            "soft_pass": soft_pass,
            "hard_pass": hard_pass,
            "accepted": soft_pass and hard_pass,
        }
    )

table(rows)

,configuration,soft_limit,hard_limit,soft_acc_T10,hard_acc,soft_pass,hard_pass,accepted
0,shared min_acc_limit=0.50,0.5,0.5,0.719581,0.75,True,True,True
1,"separate soft=0.50, hard=1.00",0.5,1.0,0.719581,0.75,True,False,False


## Mean versus min aggregation for multi-label safety

The LunarLander Rashomon safety data often uses multi-label targets: more than one action can be valid in a state. For multi-label verification, `aggregation="mean"` averages per-state certificates, while `aggregation="min"` requires every state to pass.

If your statement is "every certified state keeps at least one valid action," use `aggregation="min"` and a strict hard limit.

In [7]:
multi_label_targets = torch.tensor(
    [
        [1.0, 0.0],
        [1.0, 0.0],
        [1.0, 0.0],
        [1.0, 0.0],
    ]
)

multi_rows = []
for aggregation in ["mean", "min"]:
    multi_rows.append(
        {
            "aggregation": aggregation,
            "hard_accuracy": float(
                verify.bound_multi_label_accuracy(
                    logits,
                    multi_label_targets,
                    lower=True,
                    aggregation=aggregation,
                )
            ),
            "soft_accuracy_T10": float(
                verify.bound_multi_label_soft_accuracy(
                    logits,
                    multi_label_targets,
                    T=10,
                    lower=True,
                    aggregation=aggregation,
                )
            ),
        }
    )

table(multi_rows)

,aggregation,hard_accuracy,soft_accuracy_T10
0,mean,0.75,0.719581
1,min,0.00,0.268941


## Concrete `IntervalTrainer` example

Here are two `IntervalTrainer` objects with identical settings except for the accuracy-limit configuration. The first uses a single shared `min_acc_limit`. The second relaxes the soft optimization target while keeping the hard certificate gate strict.

In [ ]:
import copy

from src.trainer.IntervalTrainer import IntervalTrainer


COMMON_INTERVAL_TRAINER_SETTINGS = {
    "projection_strategy": "closest",
    "n_certificate_samples": 256,
    "min_acc_increment": 0.0,  # Use exact requested limits.
    "T": 10,
    "paradigm": "TIL",
    "seed": 42,
    # Forwarded to interval_utils.compute_rashomon_set via rashomon_kwargs.
    "n_iters": 5000,
    "batch_size": 500,
    "certificate_samples": 1000,
    "checkpoint": 100,
    "aggregation": "min",
}

torch.manual_seed(0)
base_actor = torch.nn.Sequential(
    torch.nn.Linear(8, 16),
    torch.nn.ReLU(),
    torch.nn.Linear(16, 4),
)

shared_limit_trainer = IntervalTrainer(
    model=copy.deepcopy(base_actor),
    min_acc_limit=0.50,
    **COMMON_INTERVAL_TRAINER_SETTINGS,
)

split_limit_trainer = IntervalTrainer(
    model=copy.deepcopy(base_actor),
    min_acc_limit=0.50,       # Fallback only; explicit values below take precedence.
    min_soft_acc_limit=0.50,  # Differentiable optimization constraint.
    min_hard_acc_limit=1.00,  # Sound hard certificate gate.
    **COMMON_INTERVAL_TRAINER_SETTINGS,
)

assert shared_limit_trainer.rashomon_kwargs == split_limit_trainer.rashomon_kwargs
assert shared_limit_trainer.min_acc_increment == split_limit_trainer.min_acc_increment

table(
    [
        {
            "trainer": "shared limit",
            "min_acc_limit": shared_limit_trainer.min_acc_limit,
            "min_soft_acc_limit": shared_limit_trainer.min_soft_acc_limit,
            "min_hard_acc_limit": shared_limit_trainer.min_hard_acc_limit,
            "min_acc_increment": shared_limit_trainer.min_acc_increment,
            "aggregation": shared_limit_trainer.rashomon_kwargs["aggregation"],
            "n_iters": shared_limit_trainer.rashomon_kwargs["n_iters"],
        },
        {
            "trainer": "separate limits",
            "min_acc_limit": split_limit_trainer.min_acc_limit,
            "min_soft_acc_limit": split_limit_trainer.min_soft_acc_limit,
            "min_hard_acc_limit": split_limit_trainer.min_hard_acc_limit,
            "min_acc_increment": split_limit_trainer.min_acc_increment,
            "aggregation": split_limit_trainer.rashomon_kwargs["aggregation"],
            "n_iters": split_limit_trainer.rashomon_kwargs["n_iters"],
        },
    ]
)

## Drop-in Rashomon patterns

Lower-level API, shared limit. This sets both soft and hard limits to `0.50`:

```python
bounded_models, certificates = interval_utils.compute_rashomon_set(
    model,
    rashomon_dataset,
    min_acc_limit=0.50,
    aggregation="mean",  # or "min" for all-states safety in multi-label mode
    multi_label=True,
)
```

Lower-level API, separate limits. This keeps the soft optimization target relaxed while requiring a strict hard certificate:

```python
bounded_models, certificates = interval_utils.compute_rashomon_set(
    model,
    rashomon_dataset,
    min_acc_limit=0.50,       # fallback only; explicit values below take precedence
    min_soft_acc_limit=0.50,  # differentiable optimization constraint
    min_hard_acc_limit=1.00,  # sound hard certificate gate
    aggregation="min",
    multi_label=True,
)
```

`IntervalTrainer` run pattern. Reuse the same compute settings for both trainers:

```python
COMPUTE_RASHOMON_SETTINGS = {
    "dataset": rashomon_dataset,
    "multi_label": True,
    "aggregation": "min",
}

shared_limit_trainer.compute_rashomon_set(**COMPUTE_RASHOMON_SETTINGS)
split_limit_trainer.compute_rashomon_set(**COMPUTE_RASHOMON_SETTINGS)
```

The mental model: `min_soft_acc_limit` is there to make optimization practical; `min_hard_acc_limit` is the soundness requirement you use to decide whether a Rashomon box is acceptable.